# MCAM Server

Loads the multi-collection ChromaDB from Hugging Face, embeds query images, retrieves + reranks AAT keywords, and serves everything via a FastAPI endpoint.

**New features:**
- Swap collection / embedding model in one config cell
- Filter results by facet or hierarchy
- `/collections` and `/facets` discovery endpoints

In [ ]:
# %%capture
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    NGROK_TOKEN = os.environ.get('NGROK_TOKEN')

!pip install torch transformers chromadb datasets qwen-vl-utils hf_xet langchain-chroma langchain-core Pillow huggingface-hub python-dotenv fastapi uvicorn pyngrok nest-asyncio python-multipart sentencepiece open_clip_torch httpx

In [2]:
from huggingface_hub import login


if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Logged in to Hugging Face')
else:
    print('HF_TOKEN not set — add it to Colab secrets or environment')


if NGROK_TOKEN:
    print('NGROK_TOKEN loaded')
else:
    print('NGROK_TOKEN not set — add it to Colab secrets or environment')

Logged in to Hugging Face
NGROK_TOKEN loaded


# (Optional) Flash Attention

Requires Ampere or newer GPU. **T4 does NOT support Flash Attention** — skip this cell on T4.

In [3]:
!nvidia-smi

# Uncomment below ONLY on Ampere+ GPU (L4, A100, etc.)
# !pip install ninja
# !pip install flash-attn --no-build-isolation

Thu Apr 16 11:48:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Configuration

Change `COLLECTION_NAME` and `EMBEDDING_TYPE` to switch which collection and model the server uses. They must match — e.g. a collection embedded with Qwen needs `EMBEDDING_TYPE = "qwen"` so query images are embedded in the same space.

In [ ]:
# ── Active collection ──
COLLECTION_NAME = "aat_terms_siglip"
EMBEDDING_TYPE  = "openclip"   # must match what was used to embed the collection

# ── Model IDs (must match the embedding notebook) ──
EMBEDDING_MODELS = {
    "qwen":     {"model_id": "Qwen/Qwen3-VL-Embedding-2B"},
    "openclip": {"model_id": "ViT-SO400M-14-SigLIP-384", "pretrained": "webli"},
}

# ── HF dataset repo containing the VDB ──
VDB_REPO = "KeeganCarey/mcam-vdb"

# ── Retrieval pool size — how many extra candidates to fetch for reranking ──
# Higher = better recall (reranker sees more candidates) but slightly slower reranking.
# Standard mode: k = term_count * RERANK_POOL_MULT, fetch_k = k * 3
# Per-hierarchy: k = count * RERANK_POOL_MULT, fetch_k = k * 3
RERANK_POOL_MULT = 5

# ── Frontend ──
FORCE_REBUILD_FRONTEND = True  # Set to False to skip rebuild when dist/ already exists
!rm -rf /content/Mills-Museum/src/frontend/web/dist

# Setup Paths

In [5]:
import os, sys, subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

# ── Change this to match the branch you're working on ──
BRANCH = "cleanup/git-rebase"

if IN_COLAB:
    repo_path = "/content/Mills-Museum"
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/cs2535-oakhoury-2026spring/Mills-Museum.git", repo_path])
    else:
        subprocess.run(["git", "fetch", "origin"], cwd=repo_path)
        subprocess.run(["git", "checkout", BRANCH], cwd=repo_path)
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=repo_path)
else:
    repo_path = str(Path(os.getcwd()).parent) if Path(os.getcwd()).name == "colab" else os.getcwd()
    if not os.path.exists(os.path.join(repo_path, "src")):
        repo_path = os.getcwd()

# Build the frontend (Vite → dist/)
frontend_dir = os.path.join(repo_path, "src", "frontend", "web")
dist_dir = os.path.join(frontend_dir, "dist")
if not os.path.isdir(dist_dir):
    print("Building frontend...")
    subprocess.run(["npm", "install"], cwd=frontend_dir, check=True)
    subprocess.run(["npx", "vite", "build"], cwd=frontend_dir, check=True)
    print("Frontend built!")
else:
    print("Frontend dist/ already exists, skipping build")

VDB_PATH = snapshot_download(repo_id=VDB_REPO, repo_type="dataset")

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Branch:      {BRANCH}")
print(f"Repo path:   {repo_path}")
print(f"VDB path:    {VDB_PATH}")

Building frontend...
Frontend built!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Environment: Colab
Branch:      cleanup/git-rebase
Repo path:   /content/Mills-Museum
VDB path:    /root/.cache/huggingface/hub/datasets--KeeganCarey--mcam-vdb/snapshots/5b357eff536ffd09b2bae6a5cb4bb8855c872be9


# Load Vector Store

Opens the ChromaDB VDB, loads the configured collection, and pre-computes the list of available facets & hierarchies for the `/facets` endpoint.

In [6]:
import chromadb
from langchain_chroma import Chroma

# Raw client — used for listing collections and facet discovery
_chroma_client = chromadb.PersistentClient(path=VDB_PATH)
available_collections = [c.name for c in _chroma_client.list_collections()]
print(f"Available collections: {available_collections}")

assert COLLECTION_NAME in available_collections, (
    f"Collection '{COLLECTION_NAME}' not found. Available: {available_collections}"
)

# LangChain wrapper — gives us MMR search
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=VDB_PATH,
    collection_metadata={"hnsw:space": "cosine"},
)
print(f"Loaded '{COLLECTION_NAME}' — {vectorstore._collection.count()} documents")

# Pre-compute facets and hierarchies from metadata
_all_meta = vectorstore._collection.get(include=["metadatas"])
AVAILABLE_FACETS = sorted(set(
    m.get("facet", "") for m in _all_meta["metadatas"] if m.get("facet")
))
AVAILABLE_HIERARCHIES = sorted(set(
    m.get("hierarchy", "") for m in _all_meta["metadatas"] if m.get("hierarchy")
))
del _all_meta

print(f"Facets ({len(AVAILABLE_FACETS)}): {AVAILABLE_FACETS}")
print(f"Hierarchies ({len(AVAILABLE_HIERARCHIES)}): {AVAILABLE_HIERARCHIES}")

Available collections: ['aat_terms_qwen2b', 'aat_terms_siglip']
Loaded 'aat_terms_qwen2b' — 19888 documents
Facets (12): ['D.DG', 'D.DL', 'F.FL', 'H.HG', 'H.HL', 'K.KM', 'K.KQ', 'K.KT', 'M.MT', 'V.PJ', 'V.R', 'V.T']
Hierarchies (12): ['Built Environment', 'Color', 'Components', 'Design Elements', 'Events', 'Furnishings and Equipment', 'Living Organisms', 'Materials', 'People', 'Physical and Mental Activities', 'Processes and Techniques', 'Styles and Periods']


# Load Embedding Model

Loads the embedding model that matches `EMBEDDING_TYPE`. This is used at query time to embed the uploaded image into the same vector space as the stored terms.

In [ ]:
import torch
import sys
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

try:
    import flash_attn
    ATTN_IMPL = "flash_attention_2"
    print("Flash Attention available")
except ImportError:
    ATTN_IMPL = "sdpa"
    print("Using SDPA attention")

embed_cfg = EMBEDDING_MODELS[EMBEDDING_TYPE]
model_id  = embed_cfg["model_id"]

if EMBEDDING_TYPE == "qwen":
    from huggingface_hub import hf_hub_download
    import importlib.util

    _script = hf_hub_download(repo_id=model_id, filename="scripts/qwen3_vl_embedding.py")
    _spec = importlib.util.spec_from_file_location("qwen3_vl_embedding", _script)
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules["qwen3_vl_embedding"] = _mod
    _spec.loader.exec_module(_mod)

    embedding_model = _mod.Qwen3VLEmbedder(
        model_name_or_path=model_id,
        dtype=torch.float16 if device == "cuda" else torch.float32,
        attn_implementation=ATTN_IMPL,
    )

    def embed_image(image):
        features = embedding_model.process([{"image": image, "text": ""}])
        features = torch.nn.functional.normalize(features, p=2, dim=1)
        return features.cpu().float().squeeze().tolist()

    def embed_text(text):
        features = embedding_model.process([{"text": text}])
        features = torch.nn.functional.normalize(features, p=2, dim=1)
        return features.cpu().float().squeeze().tolist()

elif EMBEDDING_TYPE == "openclip":
    import open_clip

    pretrained = embed_cfg.get("pretrained", "webli")
    _oc_model, _, _oc_preprocess = open_clip.create_model_and_transforms(
        model_id, pretrained=pretrained, device=device,
    )
    _oc_model.eval()
    _oc_tokenizer = open_clip.get_tokenizer(model_id)

    def embed_image(image):
        img_tensor = _oc_preprocess(image).unsqueeze(0).to(device)
        with torch.no_grad(), torch.amp.autocast(device):
            features = _oc_model.encode_image(img_tensor)
            features = torch.nn.functional.normalize(features, p=2, dim=1)
        return features.cpu().float().squeeze().tolist()

    def embed_text(text):
        tokens = _oc_tokenizer([text]).to(device)
        with torch.no_grad(), torch.amp.autocast(device):
            features = _oc_model.encode_text(tokens)
            features = torch.nn.functional.normalize(features, p=2, dim=1)
        return features.cpu().float().squeeze().tolist()

else:
    raise ValueError(f"Unknown EMBEDDING_TYPE: {EMBEDDING_TYPE}")

# Release temporary loading artifacts (does NOT affect the loaded model weights)
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

print(f"Loaded {EMBEDDING_TYPE} embedding model: {model_id}")
if device == "cuda":
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM after embedder: {allocated:.1f} GB allocated, {reserved:.1f} GB reserved")

# Load Reranker

Loads the Qwen3-VL-Reranker-2B natively via Transformers (with the classifier head for proper scoring). The GGUF approach cannot use the model's `cls.output.weight` tensor, which is required for accurate relevance scores.

In [ ]:
from huggingface_hub import hf_hub_download
import importlib.util

RERANKER_MODEL_ID = "Qwen/Qwen3-VL-Reranker-2B"

# Download and load the official Qwen3VLReranker wrapper script
_reranker_script = hf_hub_download(repo_id=RERANKER_MODEL_ID, filename="scripts/qwen3_vl_reranker.py")
_reranker_spec = importlib.util.spec_from_file_location("qwen3_vl_reranker", _reranker_script)
_reranker_mod = importlib.util.module_from_spec(_reranker_spec)
sys.modules["qwen3_vl_reranker"] = _reranker_mod
_reranker_spec.loader.exec_module(_reranker_mod)

reranking_model = _reranker_mod.Qwen3VLReranker(
    model_name_or_path=RERANKER_MODEL_ID,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    attn_implementation=ATTN_IMPL,
)

RERANKER_INSTRUCTION = (
    "You are an art cataloguing assistant. Given an artwork image, "
    "rank the following Art & Architecture Thesaurus (AAT) terms by how well each "
    "describes the visual content, subject matter, style, technique, or materials "
    "visible in the artwork. Prefer specific, visually grounded terms over generic ones."
)

print(f"Loaded native reranker: {RERANKER_MODEL_ID}")

import torch
import torch._dynamo

# Add this right before compiling
torch._dynamo.config.capture_scalar_outputs = True

# compile model on gpu
if isinstance(reranking_model, torch.nn.Module):
    reranking_model = torch.compile(reranking_model)
elif hasattr(reranking_model, 'model'):
    # This is the most common scenario for custom wrapper scripts
    reranking_model.model = torch.compile(reranking_model.model)
    print("compiled in-wrapper model")
else:
    print("Warning: Could not find the base nn.Module to compile. Running uncompiled.")

print("Model compilation configured!")

# Release temporary loading artifacts before llama-server claims VRAM
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved() / 1024**3
    total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM after reranker: {allocated:.1f} GB allocated, {reserved:.1f} GB reserved, {total:.1f} GB total")
    print(f"Available for llama-server: ~{total - reserved:.1f} GB")

# Load Captioner

Installs `llama-cpp-python` from a prebuilt CUDA wheel (~30s, no compilation),
downloads **LLaVA v1.6 Mistral 7B** in Q4_K_M GGUF (~4 GB + ~0.6 GB mmproj)
and launches the built-in OpenAI-compatible server on port 8081.

LLaVA v1.6 uses the well-supported `llava-1-6` chat format handler in
llama-cpp-python and produces better descriptions than v1.5. In MCAM
testing, LLaVA v1.6 outperformed Qwen3-VL-4B-Instruct on art descriptions
(Qwen hallucinated artist attributions) despite Qwen scoring higher on
general VLM benchmarks — so LLaVA is the known-good baseline here.

In [ ]:
import subprocess, time, sys
import requests as _requests
from huggingface_hub import hf_hub_download

# ── Install llama-cpp-python with prebuilt CUDA wheel (~30s, no compilation) ──
print("Installing llama-cpp-python (prebuilt CUDA wheel)...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "llama-cpp-python[server]",
        "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124",
        "-q",
    ],
    check=True,
)
print("llama-cpp-python installed!")

# ── Download GGUF model + mmproj ──
# LLaVA v1.6 Mistral 7B — proven vision model with native llava-1-6 chat format
CAPTIONER_REPO = "cjpais/llava-1.6-mistral-7b-gguf"
MODEL_FILE = "llava-v1.6-mistral-7b.Q4_K_M.gguf"
MMPROJ_FILE = "mmproj-model-f16.gguf"
CAPTIONER_PORT = 8081
CAPTIONER_MODEL_NAME = "llava-v1.6-mistral-7b"

model_path = hf_hub_download(repo_id=CAPTIONER_REPO, filename=MODEL_FILE)
mmproj_path = hf_hub_download(repo_id=CAPTIONER_REPO, filename=MMPROJ_FILE)
print(f"Model: {model_path}")
print(f"MMProj: {mmproj_path}")

# ── Launch llama_cpp.server as a background subprocess ──
CAPTIONER_AVAILABLE = False
_captioner_proc = None

try:
    _captioner_proc = subprocess.Popen(
        [
            sys.executable, "-m", "llama_cpp.server",
            "--model", model_path,
            "--clip_model_path", mmproj_path,
            "--chat_format", "llava-1-6",
            "--n_gpu_layers", "-1",
            "--n_ctx", "4096",
            "--host", "127.0.0.1",
            "--port", str(CAPTIONER_PORT),
            "--verbose", "False",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    CAPTIONER_URL = f"http://127.0.0.1:{CAPTIONER_PORT}"
    print(f"Waiting for llama_cpp.server on {CAPTIONER_URL}...")
    for attempt in range(90):
        time.sleep(2)
        if _captioner_proc.poll() is not None:
            stderr = _captioner_proc.stderr.read().decode("utf-8", errors="replace")
            print(f"llama_cpp.server exited with code {_captioner_proc.returncode}")
            if stderr:
                print(f"stderr (last 500 chars): ...{stderr[-500:]}")
            break
        try:
            r = _requests.get(f"{CAPTIONER_URL}/v1/models", timeout=2)
            if r.status_code == 200:
                CAPTIONER_AVAILABLE = True
                print(f"llama_cpp.server ready! (took ~{(attempt+1)*2}s)")
                break
        except _requests.ConnectionError:
            pass
    else:
        print("llama_cpp.server failed to start within 180s.")
        _captioner_proc.terminate()

except Exception as e:
    print(f"Captioner not available: {e}")
    print("The /predict-stream endpoint will be disabled; /predict still works.")

# ── Smoke test: send a tiny image to verify vision actually works ──
if CAPTIONER_AVAILABLE:
    import base64 as _b64, io as _io
    from PIL import Image as _PILTest

    print("Running vision smoke test...")
    _test_img = _PILTest.new("RGB", (64, 64), color=(128, 100, 80))
    _buf = _io.BytesIO()
    _test_img.save(_buf, format="JPEG")
    _test_b64 = _b64.b64encode(_buf.getvalue()).decode()
    _test_url = f"data:image/jpeg;base64,{_test_b64}"

    try:
        r = _requests.post(
            f"{CAPTIONER_URL}/v1/chat/completions",
            json={
                "messages": [{"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": _test_url}},
                    {"type": "text", "text": "What do you see?"},
                ]}],
                "max_tokens": 20,
            },
            timeout=120,
        )
        if r.status_code == 200:
            resp = r.json()
            text = resp["choices"][0]["message"]["content"]
            print(f"Vision smoke test PASSED: '{text[:80]}'")
        else:
            print(f"Vision smoke test FAILED: HTTP {r.status_code}")
            print(f"Response: {r.text[:500]}")
            CAPTIONER_AVAILABLE = False
    except Exception as e:
        print(f"Vision smoke test FAILED: {e}")
        if _captioner_proc and _captioner_proc.poll() is not None:
            stderr = _captioner_proc.stderr.read().decode("utf-8", errors="replace")
            print(f"Server crashed during test! stderr: ...{stderr[-500:]}")
        CAPTIONER_AVAILABLE = False

CAPTIONER_PROMPT = (
    "Describe this artwork in 2-3 sentences. Do NOT guess or name the artist, "
    "author, title, or date — only describe what is visually present. Cover the "
    "subject matter, art category (painting, sculpture, drawing, photograph, "
    "print, etc.), materials, techniques, colors, style period, and any notable "
    "visual elements. Be specific and use art terminology."
)

if CAPTIONER_AVAILABLE:
    print(f"\nCaptioner: LLaVA v1.6 Mistral 7B Q4_K_M via llama_cpp.server (port {CAPTIONER_PORT})")
    if device == "cuda":
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved  = torch.cuda.memory_reserved() / 1024**3
        total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"PyTorch VRAM: {allocated:.1f} GB allocated, {reserved:.1f} GB reserved")
else:
    print("\nCaptioner disabled — /predict will still work without query expansion.")

# API Server

Endpoints:
- `POST /predict` — upload an image, get initial (unscored) AAT keywords + a `job_id`. Reranking runs in the background.
- `POST /predict-stream` — same as `/predict` but streams a VLM artwork description via SSE first, then uses it for query expansion before returning keywords. Only available when the captioner model is loaded.
- `GET /predict-status/{job_id}` — poll for reranking progress. Returns scored keywords as they complete.
- `GET /collections` — list all collections in the VDB.
- `GET /facets` — list available facets and hierarchies for filtering.
- `GET /` — serve the frontend.

In [ ]:
import uvicorn
import nest_asyncio
import threading
import uuid
import time
import math
import json as _json
import asyncio
import base64
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.staticfiles import StaticFiles
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok, conf
from PIL import Image
from typing import Optional
import io
import os
import httpx

nest_asyncio.apply()
conf.get_default().auth_token = NGROK_TOKEN

# ── Job store for progressive reranking ──
_jobs = {}
_jobs_lock = threading.Lock()
_reranker_lock = threading.Lock()
JOB_TTL_SECONDS = 600


def _sweep_expired_jobs():
    """Remove jobs older than JOB_TTL_SECONDS."""
    now = time.time()
    with _jobs_lock:
        expired = [jid for jid, j in _jobs.items() if now - j["created_at"] > JOB_TTL_SECONDS]
        for jid in expired:
            del _jobs[jid]


def _check_captioner_alive():
    """Check if the captioner subprocess is still running. If it crashed, log stderr and disable."""
    global CAPTIONER_AVAILABLE
    if not CAPTIONER_AVAILABLE or _captioner_proc is None:
        return False
    if _captioner_proc.poll() is not None:
        CAPTIONER_AVAILABLE = False
        stderr = ""
        try:
            stderr = _captioner_proc.stderr.read().decode("utf-8", errors="replace")
        except Exception:
            pass
        print(f"[CAPTIONER CRASHED] exit code {_captioner_proc.returncode}")
        if stderr:
            print(f"  stderr (last 800 chars): ...{stderr[-800:]}")
        return False
    return True


def _rerank_job(job_id, image, input_docs, labels, doc_metadata):
    """Background thread: scores all documents at once via the native reranker,
    then progressively updates job state so the frontend can poll."""
    try:
        with _reranker_lock:
            rerank_inputs = {
                "instruction": RERANKER_INSTRUCTION,
                "query": {"image": image},
                "documents": input_docs,
                "fps": 1.0,
            }
            scores = reranking_model.process(rerank_inputs)

        # Apply scores to job keywords
        with _jobs_lock:
            job = _jobs.get(job_id)
            if job is None:
                return
            for i, score in enumerate(scores):
                job["keywords"][i]["score"] = round(float(score) * 100, 1)
            job["completed"] = len(scores)
            job["status"] = "done"
            job["keywords"].sort(
                key=lambda k: k["score"] if k["score"] is not None else -1,
                reverse=True,
            )

    except Exception as e:
        import traceback
        traceback.print_exc()
        with _jobs_lock:
            job = _jobs.get(job_id)
            if job:
                job["status"] = "error"
                job["error"] = str(e)


def _pil_to_base64_jpeg(image: Image.Image, quality: int = 85) -> str:
    """Convert a PIL image to a base64-encoded JPEG data URL."""
    buf = io.BytesIO()
    image.save(buf, format="JPEG", quality=quality)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"


app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Serve the Vite production build
DIST_PATH = os.path.join(repo_path, "src", "frontend", "web", "dist")


@app.get("/collections")
async def list_collections():
    """List all collections in the VDB."""
    return {
        "active": COLLECTION_NAME,
        "available": available_collections,
    }


@app.get("/facets")
async def list_facets():
    """List facets and hierarchies available for filtering."""
    return {
        "facets": AVAILABLE_FACETS,
        "hierarchies": AVAILABLE_HIERARCHIES,
    }


def _retrieve_single_hierarchy(query_embedding, hierarchy_name, count, lambda_mult):
    """Run one MMR query for a single hierarchy. Returns (labels, input_docs, doc_metadata)."""
    pool_k = count * RERANK_POOL_MULT
    docs = vectorstore.max_marginal_relevance_search_by_vector(
        embedding=query_embedding,
        k=pool_k,
        fetch_k=pool_k * 3,
        lambda_mult=lambda_mult,
        filter={"hierarchy": hierarchy_name},
    )
    labels = []
    input_docs = []
    doc_metadata = []
    seen_terms = set()
    for doc in docs:
        term = doc.metadata.get("preferred_term", doc.page_content.split(":")[0])
        if term in seen_terms:
            continue
        seen_terms.add(term)
        labels.append(term)
        input_docs.append({"text": doc.page_content})
        doc_metadata.append(doc.metadata)
        if len(labels) >= count:
            break
    return labels, input_docs, doc_metadata


def _retrieve_per_hierarchy(query_embedding, counts_map, lambda_mult):
    """Run separate MMR queries per hierarchy **in parallel** and combine results.

    Parameters
    ----------
    query_embedding : list[float]
    counts_map : dict[str, int]  – e.g. {"Materials": 3, "Color": 2}
    lambda_mult : float – MMR diversity parameter

    Returns
    -------
    labels, input_docs, doc_metadata – combined across all hierarchies.
    """
    # Filter out zero-count hierarchies
    active = {h: c for h, c in counts_map.items() if c > 0}
    if not active:
        return [], [], []

    # Run all hierarchy queries in parallel
    with ThreadPoolExecutor(max_workers=min(len(active), 8)) as pool:
        futures = {
            h: pool.submit(_retrieve_single_hierarchy, query_embedding, h, c, lambda_mult)
            for h, c in active.items()
        }

    # Combine results, deduplicating across hierarchies
    labels = []
    input_docs = []
    doc_metadata = []
    seen_terms = set()

    for hierarchy_name in active:
        h_labels, h_docs, h_meta = futures[hierarchy_name].result()
        for lbl, doc, meta in zip(h_labels, h_docs, h_meta):
            if lbl not in seen_terms:
                seen_terms.add(lbl)
                labels.append(lbl)
                input_docs.append(doc)
                doc_metadata.append(meta)

    return labels, input_docs, doc_metadata


def _retrieve_standard(query_embedding, term_count, mmr_lambda, where_filter=None):
    """Standard MMR retrieval with wide candidate pool for reranking."""
    if term_count <= 0:
        return [], [], []
    pool_k = term_count * RERANK_POOL_MULT
    mmr_docs = vectorstore.max_marginal_relevance_search_by_vector(
        embedding=query_embedding,
        k=pool_k,
        fetch_k=pool_k * 3,
        lambda_mult=mmr_lambda,
        filter=where_filter,
    )

    seen_terms = set()
    labels = []
    input_docs = []
    doc_metadata = []
    for doc in mmr_docs:
        term = doc.metadata.get("preferred_term", doc.page_content.split(":")[0])
        if term in seen_terms:
            continue
        seen_terms.add(term)
        labels.append(term)
        input_docs.append({"text": doc.page_content})
        doc_metadata.append(doc.metadata)
        if len(labels) >= term_count:
            break

    return labels, input_docs, doc_metadata


def _merge_retrieval_results(*result_tuples):
    """Merge multiple (labels, input_docs, doc_metadata) tuples, deduplicating by label."""
    seen = set()
    labels, input_docs, doc_metadata = [], [], []
    for r_labels, r_docs, r_meta in result_tuples:
        for lbl, doc, meta in zip(r_labels, r_docs, r_meta):
            if lbl not in seen:
                seen.add(lbl)
                labels.append(lbl)
                input_docs.append(doc)
                doc_metadata.append(meta)
    return labels, input_docs, doc_metadata


def _parse_hierarchy_counts(raw):
    """Parse + validate a hierarchy_counts JSON string. Raises ValueError on bad input."""
    counts_map = _json.loads(raw)
    bad_keys = [k for k in counts_map if k not in AVAILABLE_HIERARCHIES]
    if bad_keys:
        raise ValueError(f"Unknown hierarchies: {bad_keys}")
    for k, v in counts_map.items():
        if not isinstance(v, (int, float)) or v < 0 or v > 50:
            raise ValueError(f"Count for '{k}' must be 0-50, got {v}")
        counts_map[k] = int(v)
    return counts_map


def _build_where_filter(facets, hierarchies):
    """Build a Chroma where-filter from comma-separated facets + hierarchies strings."""
    conditions = []
    if facets:
        conditions.append({"facet": {"$in": [f.strip() for f in facets.split(",")]}})
    if hierarchies:
        conditions.append({"hierarchy": {"$in": [h.strip() for h in hierarchies.split(",")]}})
    if len(conditions) == 1:
        return conditions[0]
    if len(conditions) > 1:
        return {"$and": conditions}
    return None


def _do_retrieval(query_embedding, term_count, mmr_lambda, hierarchy_counts_raw=None,
                  facets=None, hierarchies=None):
    """Shared single-query retrieval for /predict (no text-query expansion)."""
    if hierarchy_counts_raw:
        counts_map = _parse_hierarchy_counts(hierarchy_counts_raw)
        return _retrieve_per_hierarchy(query_embedding, counts_map, mmr_lambda)
    return _retrieve_standard(query_embedding, term_count, mmr_lambda,
                              _build_where_filter(facets, hierarchies))


def _create_job_and_rerank(image, labels, input_docs, doc_metadata):
    """Create a reranking job and start the background thread. Returns (job_id, initial_keywords)."""
    job_id = str(uuid.uuid4())
    initial_keywords = [
        {
            "label":      label,
            "score":      None,
            "facet":      meta.get("facet", ""),
            "hierarchy":  meta.get("hierarchy", ""),
            "subject_id": meta.get("subject_id", ""),
            "scope_note": meta.get("scope_note", ""),
        }
        for label, meta in zip(labels, doc_metadata)
    ]

    with _jobs_lock:
        _jobs[job_id] = {
            "status": "reranking",
            "total": len(input_docs),
            "completed": 0,
            "keywords": initial_keywords,
            "error": None,
            "created_at": time.time(),
        }

    thread = threading.Thread(
        target=_rerank_job,
        args=(job_id, image, input_docs, labels, doc_metadata),
        daemon=True,
    )
    thread.start()

    return job_id, initial_keywords


@app.post("/predict")
async def predict(
    file: UploadFile = File(...),
    term_count: Optional[int] = Form(default=None),
    facets: Optional[str] = Form(default=None),
    hierarchies: Optional[str] = Form(default=None),
    hierarchy_counts: Optional[str] = Form(default=None),
    lambda_mult: float = Form(...),
):
    if term_count is None and not hierarchy_counts:
        return JSONResponse(
            status_code=422,
            content={"error": "Must provide term_count or hierarchy_counts"},
        )

    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    mmr_lambda = max(0.0, min(1.0, float(lambda_mult)))

    query_embedding = embed_image(image)

    try:
        labels, input_docs, doc_metadata = _do_retrieval(
            query_embedding, term_count, mmr_lambda,
            hierarchy_counts_raw=hierarchy_counts,
            facets=facets, hierarchies=hierarchies,
        )
    except (ValueError, _json.JSONDecodeError) as e:
        return JSONResponse(status_code=422, content={"error": str(e)})

    job_id, initial_keywords = _create_job_and_rerank(image, labels, input_docs, doc_metadata)
    return {"job_id": job_id, "keywords": initial_keywords}


@app.post("/predict-stream")
async def predict_stream(
    file: UploadFile = File(...),
    term_count: Optional[int] = Form(default=None),
    facets: Optional[str] = Form(default=None),
    hierarchies: Optional[str] = Form(default=None),
    hierarchy_counts: Optional[str] = Form(default=None),
    lambda_mult: float = Form(...),
    query_bias: float = Form(...),
):
    """Stream a VLM artwork description via SSE, then return keywords with query expansion.

    `query_bias` splits term_count between the image-based and text-based searches:
    0.0 = 100% image, 1.0 = 100% description. On uneven splits, image rounds UP
    (math.ceil) and text gets the remainder.
    """
    # Check if captioner is still alive (may have crashed on a previous request)
    if not _check_captioner_alive():
        return JSONResponse(status_code=503, content={"error": "Captioner model not loaded"})

    if term_count is None and not hierarchy_counts:
        return JSONResponse(
            status_code=422,
            content={"error": "Must provide term_count or hierarchy_counts"},
        )

    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    mmr_lambda = max(0.0, min(1.0, float(lambda_mult)))
    bias = max(0.0, min(1.0, float(query_bias)))

    # Convert image to base64 for the llama-server API
    image_data_url = _pil_to_base64_jpeg(image)

    async def event_generator():
        loop = asyncio.get_event_loop()

        # ── Phase 1: Stream artwork description via llama_cpp.server ──
        yield f"data: {_json.dumps({'type': 'status', 'message': 'Describing artwork...'})}\n\n"

        description_chunks = []
        try:
            async with httpx.AsyncClient(timeout=httpx.Timeout(120.0)) as client:
                async with client.stream(
                    "POST",
                    f"{CAPTIONER_URL}/v1/chat/completions",
                    json={
                        "model": CAPTIONER_MODEL_NAME,
                        "messages": [
                            {
                                "role": "user",
                                "content": [
                                    {
                                        "type": "image_url",
                                        "image_url": {"url": image_data_url},
                                    },
                                    {
                                        "type": "text",
                                        "text": CAPTIONER_PROMPT,
                                    },
                                ],
                            }
                        ],
                        "max_tokens": 200,
                        "stream": True,
                    },
                ) as resp:
                    resp.raise_for_status()
                    async for line in resp.aiter_lines():
                        if not line.startswith("data: "):
                            continue
                        payload = line[6:].strip()
                        if payload == "[DONE]":
                            break
                        try:
                            chunk = _json.loads(payload)
                            delta = chunk.get("choices", [{}])[0].get("delta", {})
                            token = delta.get("content", "")
                            if token:
                                description_chunks.append(token)
                                yield f"data: {_json.dumps({'type': 'description', 'token': token})}\n\n"
                        except (_json.JSONDecodeError, IndexError, KeyError):
                            continue

        except Exception as e:
            print(f"Captioner streaming error: {e}")
            _check_captioner_alive()  # Check if it crashed and log stderr
            # Continue without description — retrieval still works via image embedding

        full_description = "".join(description_chunks).strip()
        yield f"data: {_json.dumps({'type': 'description_done', 'full': full_description})}\n\n"

        # ── Phase 2: Image embedding + text query expansion + retrieval ──
        yield f"data: {_json.dumps({'type': 'status', 'message': 'Retrieving keywords...'})}\n\n"

        query_embedding = await loop.run_in_executor(None, embed_image, image)

        # Parse per-hierarchy counts up front (validates JSON + keys)
        counts_map = None
        if hierarchy_counts:
            try:
                counts_map = _parse_hierarchy_counts(hierarchy_counts)
            except (ValueError, _json.JSONDecodeError) as e:
                yield f"data: {_json.dumps({'type': 'error', 'message': str(e)})}\n\n"
                return

        # Split counts between image and text searches based on bias.
        # math.ceil on image side → image keeps the extra on uneven splits.
        if counts_map is not None:
            img_counts  = {h: math.ceil((1 - bias) * c) for h, c in counts_map.items()}
            text_counts = {h: max(0, c - img_counts[h]) for h, c in counts_map.items()}
            img_has_any  = any(v > 0 for v in img_counts.values())
            text_has_any = any(v > 0 for v in text_counts.values())
        else:
            img_count  = math.ceil((1 - bias) * term_count)
            text_count = term_count - img_count
            img_has_any, text_has_any = (img_count > 0), (text_count > 0)

        # Image retrieval
        if counts_map is not None:
            img_results = await loop.run_in_executor(
                None, _retrieve_per_hierarchy, query_embedding, img_counts, mmr_lambda,
            ) if img_has_any else ([], [], [])
        else:
            where_filter = _build_where_filter(facets, hierarchies)
            img_results = await loop.run_in_executor(
                None, _retrieve_standard, query_embedding, img_count, mmr_lambda, where_filter,
            ) if img_has_any else ([], [], [])

        # Text retrieval (only if description exists AND text_count > 0)
        if full_description and text_has_any:
            text_embedding = await loop.run_in_executor(None, embed_text, full_description)
            if counts_map is not None:
                text_results = await loop.run_in_executor(
                    None, _retrieve_per_hierarchy, text_embedding, text_counts, mmr_lambda,
                )
            else:
                text_results = await loop.run_in_executor(
                    None, _retrieve_standard, text_embedding, text_count, mmr_lambda, None,
                )
            labels, input_docs, doc_metadata = _merge_retrieval_results(img_results, text_results)
        else:
            text_results = ([], [], [])
            labels, input_docs, doc_metadata = img_results

        # Retrieval stats: how many came from each query + duplicates
        img_hit_count = len(img_results[0])
        text_hit_count = len(text_results[0])
        unique_count = len(labels)
        duplicate_count = img_hit_count + text_hit_count - unique_count
        retrieval_stats = {
            "image": img_hit_count,
            "description": text_hit_count,
            "duplicates": duplicate_count,
            "unique": unique_count,
        }

        # ── Phase 3: Create job + start reranking ──
        yield f"data: {_json.dumps({'type': 'status', 'message': 'Scoring keywords...'})}\n\n"

        job_id, initial_keywords = _create_job_and_rerank(image, labels, input_docs, doc_metadata)

        yield f"data: {_json.dumps({'type': 'result', 'job_id': job_id, 'keywords': initial_keywords, 'retrieval_stats': retrieval_stats})}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",
        },
    )


@app.get("/predict-status/{job_id}")
async def predict_status(job_id: str):
    """Poll for reranking progress. Returns current scores and completion count."""
    _sweep_expired_jobs()

    with _jobs_lock:
        job = _jobs.get(job_id)

    if not job:
        return JSONResponse(status_code=404, content={"error": "Job not found"})

    return {
        "status": job["status"],
        "total": job["total"],
        "completed": job["completed"],
        "keywords": job["keywords"],
        "error": job.get("error"),
    }


# Mount the built frontend as a catch-all (after API routes so they take priority)
app.mount("/", StaticFiles(directory=DIST_PATH, html=True), name="frontend")

# Launch
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")
print("Open this URL in your browser to use the app!")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()